In [58]:
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import polars as pl
import random

In [59]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_ROOT = PROJECT_ROOT / "tennis_data"

EXTRACT_ROOT = DATA_ROOT / "extracted"

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

In [60]:
# Collect all event parquet files from extracted data

event_files = sorted(
    EXTRACT_ROOT.glob(
        "*/raw_match_parquet/event_*.parquet"
    )
)


print(
    "Total number of event files:",
    len(event_files)
)

Total number of event files: 35053


In [61]:
# Group event files by match_id

event_match_files = defaultdict(list)


for file in event_files:

    match_id = file.stem.replace(
        "event_",
        ""
    )

    event_match_files[match_id].append(file)


print(
    "Number of unique event match IDs:",
    len(event_match_files)
)

Number of unique event match IDs: 16873


In [62]:
# Count snapshots available for each event match ID

event_snapshot_counts = {
    match_id: len(files)
    for match_id, files in event_match_files.items()
}


for match_id, count in list(
    event_snapshot_counts.items()
)[:10]:

    print(
        f"Match ID: {match_id} | Snapshots: {count}"
    )

Match ID: 11974053 | Snapshots: 3
Match ID: 11974066 | Snapshots: 2
Match ID: 11998445 | Snapshots: 2
Match ID: 11998446 | Snapshots: 2
Match ID: 11998447 | Snapshots: 2
Match ID: 11998448 | Snapshots: 1
Match ID: 11998449 | Snapshots: 1
Match ID: 11998450 | Snapshots: 1
Match ID: 11998451 | Snapshots: 1
Match ID: 11998456 | Snapshots: 3


In [63]:
# Find the maximum number of snapshots for one match ID

max_snapshots = max(
    event_snapshot_counts.values()
)


print(
    "Maximum number of snapshots:",
    max_snapshots
)

Maximum number of snapshots: 4


In [64]:
# Show distribution of snapshot counts across match IDs

snapshot_distribution = Counter(
    event_snapshot_counts.values()
)


for snapshots, match_count in sorted(
    snapshot_distribution.items()
):

    print(
        f"{snapshots} snapshot(s): "
        f"{match_count} match IDs"
    )

1 snapshot(s): 980 match IDs
2 snapshot(s): 13619 match IDs
3 snapshot(s): 2261 match IDs
4 snapshot(s): 13 match IDs


In [65]:
# Check the number of columns in all event files

from collections import Counter

column_count_distribution = Counter(
    len(pl.read_parquet(file).columns)
    for file in event_files
)

print(column_count_distribution)

Counter({10: 35053})


In [66]:
# Check whether all event files have the same column names

reference_columns = pl.read_parquet(event_files[0]).columns

different_column_files = []

for file in event_files:
    columns = pl.read_parquet(file).columns

    if columns != reference_columns:
        different_column_files.append(file)

print(
    "Files with different column names:",
    len(different_column_files)
)

Files with different column names: 0


In [67]:
# Check whether event files have different schemas

reference_schema = pl.read_parquet(event_files[0]).schema

different_schema_files = []

for file in event_files:
    schema = pl.read_parquet(file).schema

    if schema != reference_schema:
        different_schema_files.append(file)

print(
    "Files with different schema:",
    len(different_schema_files)
)

Files with different schema: 28952


In [68]:
# Show data types observed for each event column

column_dtypes = {
    column: set()
    for column in reference_columns
}

for file in event_files:
    schema = pl.read_parquet(file).schema

    for column, dtype in schema.items():
        column_dtypes[column].add(str(dtype))

for column, dtypes in column_dtypes.items():
    print(
        f"{column}: {sorted(dtypes)}"
    )

match_id: ['Int64']
first_to_serve: ['Int64', 'Null']
home_team_seed: ['Null', 'String']
away_team_seed: ['Null', 'String']
custom_id: ['String']
winner_code: ['Int64', 'Null']
default_period_count: ['Int64']
start_datetime: ['Int64']
match_slug: ['String']
final_result_only: ['Boolean']


In [70]:
# Define fixed data types for event columns

event_schema = {
    "match_id": pl.Int64,
    "first_to_serve": pl.Int64,
    "home_team_seed": pl.String,
    "away_team_seed": pl.String,
    "custom_id": pl.String,
    "winner_code": pl.Int64,
    "default_period_count": pl.Int64,
    "start_datetime": pl.Int64,
    "match_slug": pl.String,
    "final_result_only": pl.Boolean
}

In [71]:
# Test fixed schema on one event file

test_event_df = pl.read_parquet(
    event_files[0]
)

test_event_df = test_event_df.cast(
    event_schema,
    strict=False
)

print(test_event_df.schema)

Schema([('match_id', Int64), ('first_to_serve', Int64), ('home_team_seed', String), ('away_team_seed', String), ('custom_id', String), ('winner_code', Int64), ('default_period_count', Int64), ('start_datetime', Int64), ('match_slug', String), ('final_result_only', Boolean)])


In [72]:
# Read event files with fixed schema and add snapshot date

event_frames = []

for file in event_files:
    df = pl.read_parquet(file)

    df = df.cast(
        event_schema,
        strict=False
    )

    snapshot_date = file.parent.parent.name

    df = df.with_columns(
        pl.lit(snapshot_date)
        .str.to_date("%Y%m%d")
        .alias("snapshot_date")
    )

    event_frames.append(df)

print("Event files processed:", len(event_frames))

Event files processed: 35053


In [73]:
# Concatenate all processed event snapshots into one dataframe

event_snapshot = pl.concat(
    event_frames,
    how="vertical_relaxed"
)

print(
    "Final event shape:",
    event_snapshot.shape
)

Final event shape: (35053, 11)


In [74]:
# Convert Unix timestamp to readable match datetime

event_snapshot = event_snapshot.with_columns(
    pl.from_epoch(
        pl.col("start_datetime"),
        time_unit="s"
    ).alias("match_datetime")
)

In [94]:
# Create one chronological start time for each match

match_order = (
    event_snapshot
    .group_by("match_id")
    .agg(
        pl.col("match_datetime")
        .min()
        .alias("match_start_time")
    )
    .sort(
        ["match_start_time", "match_id"]
    )
    .with_row_index("match_order")
)

In [95]:
# Attach match order to all snapshots

event_snapshot = event_snapshot.join(
    match_order.select(
        ["match_id", "match_order", "match_start_time"]
    ),
    on="match_id",
    how="left"
)

In [96]:
# Sort matches chronologically while keeping snapshots together

event_snapshot = event_snapshot.sort(
    [
        "match_order",
        "snapshot_date"
    ]
)

In [97]:
# Check the first rows before saving

event_snapshot.select(
    [
        "match_id",
        "match_datetime",
        "snapshot_date",
        "match_order"
    ]
).head(30)

match_id,match_datetime,snapshot_date,match_order
i64,datetime[μs],date,u32
12011306,2024-01-31 12:05:00,2024-02-01,0
12017464,2024-01-31 12:05:00,2024-02-01,1
12018812,2024-01-31 12:05:00,2024-02-01,2
12021601,2024-01-31 12:05:00,2024-02-01,3
12017522,2024-01-31 12:15:00,2024-02-01,4
…,…,…,…
12018945,2024-01-31 14:05:00,2024-02-01,25
12019949,2024-01-31 14:05:00,2024-02-01,26
12017506,2024-01-31 14:10:00,2024-02-01,27


In [99]:
# Check ordering for one match with multiple snapshots

event_snapshot.filter(
    pl.col("match_id") == 11974053
).select(
    [
        "match_id",
        "match_datetime",
        "snapshot_date",
        "match_order"
    ]
)

match_id,match_datetime,snapshot_date,match_order
i64,datetime[μs],date,u32
11974053,2024-02-02 13:00:00,2024-02-01,279
11974053,2024-02-02 13:00:00,2024-02-02,279
11974053,2024-02-02 13:00:00,2024-02-03,279


In [100]:
# Remove temporary sorting columns

event_snapshot = event_snapshot.drop(
    ["match_order", "match_start_time"]
)

In [101]:
# Save the processed event dataset

processed_path = DATA_ROOT / "Data"

processed_path.mkdir(
    parents=True,
    exist_ok=True
)

event_snapshot.write_parquet(
    processed_path / "event.parquet"
)

In [102]:
# Validate the saved event parquet file

saved_event = pl.read_parquet(
    processed_path / "event.parquet"
)

print("Saved shape:", saved_event.shape)

saved_event.head(20)

Saved shape: (35053, 12)


match_id,first_to_serve,home_team_seed,away_team_seed,custom_id,winner_code,default_period_count,start_datetime,match_slug,final_result_only,snapshot_date,match_datetime
i64,i64,str,str,str,i64,i64,i64,str,bool,date,datetime[μs]
12011306,2,"""8""",null,"""RygsiDdc""",null,3,1706702700,"""passaro-marchenko""",false,2024-02-01,2024-01-31 12:05:00
12017464,1,"""Q""",null,"""iBJcsdQYc""",2,3,1706702700,"""kuhl-salkova""",false,2024-02-01,2024-01-31 12:05:00
12018812,null,"""Q""","""2""","""KMzbsLzjd""",2,3,1706702700,"""gokpinar-hardt""",false,2024-02-01,2024-01-31 12:05:00
12021601,1,"""LL""",null,"""RygsfTx""",1,3,1706702700,"""jorda-sanchis-marchenko""",false,2024-02-01,2024-01-31 12:05:00
12017522,2,"""WC""",null,"""ArQsedIb""",1,3,1706703300,"""tubello-serban""",false,2024-02-01,2024-01-31 12:15:00
…,…,…,…,…,…,…,…,…,…,…,…
12021558,2,"""LL""",null,"""aBosgmB""",2,3,1706706000,"""agamenone-travaglia""",false,2024-02-01,2024-01-31 13:00:00
12018944,1,null,null,"""LWAcsKkZc""",1,3,1706706600,"""van-impe-hietaranta""",false,2024-02-01,2024-01-31 13:10:00
12017533,1,"""1""",null,"""tZAsizrc""",1,3,1706706900,"""boisson-dodin""",false,2024-02-01,2024-01-31 13:15:00
